In [ ]:
import os
import pandas as pd

# Notebook ke liye correct path
DATA_DIR = os.path.join('..', 'Data', 'Raw')

def ingest_and_inspect_data():
    if not os.path.exists(DATA_DIR):
        print(f"Error: Folder path '{DATA_DIR}' nahi mila!")
        return

    # Folder me se saari CSV files ki list nikalna
    files = [f for f in sorted(os.listdir(DATA_DIR)) if f.endswith('.csv')]

    if not files:
        print("Koi CSV file nahi mili data/raw folder me!")
        return

    print(f"Total {len(files)} CSV files mili hain. Inspection shuru ho raha hai...\n")
    print("-" * 60)

    for file in files:
        file_path = os.path.join(DATA_DIR, file)
        print(f"\n FILE: {file}")
        print("-" * 60)

        try:
            # 1. Sab CSV read karna
            df = pd.read_csv(file_path)

            # 2. Shape print karna
            print(f" Shape (Rows, Columns): {df.shape}")

            # 3. First 5 rows print karna
            print("\n First 5 Rows:")
            print(df.head())

            # 4. Datatype print karna
            print("\n Data Types:")
            print(df.dtypes)

            # 5. Missing values check karna
            print("\n Missing Values (per column):")
            print(df.isnull().sum())

            # 6. Duplicate check karna
            print(f"\n Duplicate Rows Count: {df.duplicated().sum()}")

        except Exception as e:
            print(f"Error reading {file}: {e}")

        print("-" * 60)

# Function call
ingest_and_inspect_data()

In [ ]:
import pandas as pd

# Load Fund Master Dataset
df = pd.read_csv("../Data/Raw/01_fund_master.csv")

# Unique Fund Houses
print("=" * 50)
print("UNIQUE FUND HOUSES")
print("=" * 50)
print(df["fund_house"].unique())

# Unique Categories
print("\n" + "=" * 50)
print("UNIQUE CATEGORIES")
print("=" * 50)
print(df["category"].unique())

# Unique Sub Categories
print("\n" + "=" * 50)
print("UNIQUE SUB CATEGORIES")
print("=" * 50)
print(df["sub_category"].unique())

# Unique Risk Categories
print("\n" + "=" * 50)
print("UNIQUE RISK CATEGORIES")
print("=" * 50)
print(df["risk_category"].unique())

In [7]:
import os
import pandas as pd

# Files ke paths define karein
fund_master_path = os.path.join('..', 'Data', 'Raw', '01_fund_master.csv')
nav_history_path = os.path.join('..', 'Data', 'Raw', '02_nav_history.csv')

# CSV files load karein
df_master = pd.read_csv(fund_master_path)
df_nav = pd.read_csv(nav_history_path)

# Unique AMFI codes dono tables se nikalen
master_codes = set(df_master['amfi_code'].unique())
nav_codes = set(df_nav['amfi_code'].unique())

# Missing codes check karein (jo master me hain par nav history me nahi)
missing_codes = master_codes - nav_codes

print(f"Total Unique AMFI Codes in Fund Master: {len(master_codes)}")
print(f"Total Unique AMFI Codes in NAV History: {len(nav_codes)}")
print("-" * 50)

if len(missing_codes) == 0:
    print("✅ Success: Fund Master ke SAARE amfi_code NAV History me available hain!")
else:
    print(f"⚠️ Warning: Total {len(missing_codes)} amfi_code NAV History me missing hain.")
    print("Missing Codes ka list:", list(missing_codes)[:10], "... (Showing first 10)")

Total Unique AMFI Codes in Fund Master: 40
Total Unique AMFI Codes in NAV History: 40
--------------------------------------------------
✅ Success: Fund Master ke SAARE amfi_code NAV History me available hain!


In [8]:
import os
import pandas as pd

# Directories setup
DATA_DIR = os.path.join('..', 'Data', 'Raw')
REPORT_DIR = os.path.join('..', 'Report')

# Agar Report folder nahi hai toh bana dega
os.makedirs(REPORT_DIR, exist_ok=True)
report_file_path = os.path.join(REPORT_DIR, 'data_quality_report.md')

# Markdown content builder
markdown_content = "# Data Quality Report\n\n"

files = [f for f in sorted(os.listdir(DATA_DIR)) if f.endswith('.csv')]

for file in files:
    file_path = os.path.join(DATA_DIR, file)
    try:
        df = pd.read_csv(file_path)
        
        rows, cols = df.shape
        missing_val_summary = df.isnull().sum()
        missing_cols = missing_val_summary[missing_val_summary > 0]
        duplicates = df.duplicated().sum()
        
        markdown_content += f"## File: `{file}`\n"
        markdown_content += f"- **Rows:** {rows}\n"
        markdown_content += f"- **Columns:** {cols}\n"
        
        # Missing Values section
        markdown_content += "- **Missing Values:**\n"
        if len(missing_cols) == 0:
            markdown_content += "  - None\n"
        else:
            for col, count in missing_cols.items():
                markdown_content += f"  - `{col}`: {count}\n"
                
        # Duplicate Rows section
        markdown_content += f"- **Duplicate Rows:** {duplicates}\n\n"

    except Exception as e:
        markdown_content += f"## File: `{file}`\n"
        markdown_content += f"**Error loading file:** {e}\n\n"

# AMFI Code Validation Section
fund_master_path = os.path.join(DATA_DIR, '01_fund_master.csv')
nav_history_path = os.path.join(DATA_DIR, '02_nav_history.csv')

markdown_content += "## AMFI Code Validation\n"
if os.path.exists(fund_master_path) and os.path.exists(nav_history_path):
    df_master = pd.read_csv(fund_master_path)
    df_nav = pd.read_csv(nav_history_path)
    
    master_codes = set(df_master['amfi_code'].unique())
    nav_codes = set(df_nav['amfi_code'].unique())
    missing_codes = master_codes - nav_codes
    
    if len(missing_codes) == 0:
        markdown_content += "- **Status:** ✅ All AMFI codes in `01_fund_master.csv` are present in `02_nav_history.csv`.\n\n"
    else:
        markdown_content += f"- **Status:** ⚠️ {len(missing_codes)} AMFI codes from master are missing in NAV history.\n\n"
else:
    markdown_content += "- **Status:** Files required for validation were not found.\n\n"

# Recommendations Section
markdown_content += "## Recommendations\n"
markdown_content += "1. Impute or drop missing values depending on the column's importance before feeding into models.\n"
markdown_content += "2. Remove any duplicate rows found during the inspection step.\n"
markdown_content += "3. Standardize column names across all files to ensure consistent data pipelines.\n"

# Save the Markdown file
with open(report_file_path, 'w', encoding='utf-8') as f:
    f.write(markdown_content)

print(f"✅ Data Quality Report successfully generated at: {report_file_path}")

✅ Data Quality Report successfully generated at: ..\Report\data_quality_report.md


In [1]:
import pandas as pd

fund_master = pd.read_csv("../data/raw/01_fund_master.csv")
nav_history = pd.read_csv("../data/raw/02_nav_history.csv")

print(fund_master.columns)
print(nav_history.columns)

Index(['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category',
       'plan', 'launch_date', 'benchmark', 'expense_ratio_pct',
       'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager',
       'risk_category', 'sebi_category_code'],
      dtype='str')
Index(['amfi_code', 'date', 'nav'], dtype='str')


In [2]:
print("Unique Fund Houses:")
print(fund_master["fund_house"].unique())

print("Unique Categories:")
print(fund_master["category"].unique())

missing_codes = set(fund_master["amfi_code"]) - set(nav_history["amfi_code"])

print("Missing AMFI Codes:", len(missing_codes))
print(missing_codes)

Unique Fund Houses:
<StringArray>
[         'SBI Mutual Fund',         'HDFC Mutual Fund',
      'ICICI Prudential MF',          'Nippon India MF',
        'Kotak Mahindra MF',         'Axis Mutual Fund',
 'Aditya Birla Sun Life MF',          'UTI Mutual Fund',
           'Mirae Asset MF',          'DSP Mutual Fund']
Length: 10, dtype: str
Unique Categories:
<StringArray>
['Equity', 'Debt']
Length: 2, dtype: str
Missing AMFI Codes: 0
set()
